In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas
import numpy
import seaborn
import matplotlib.pyplot as plt

In [ ]:
# Thiết lập để pandas hiển thị tất cả các cột
pandas.set_option('display.max_columns', None)
# Thiết lập để pandas hiển thị tất cả các dòng
pandas.set_option('display.max_rows', None)

pandas.set_option('display.float_format', lambda x:'%f'%x)

In [ ]:
# Đọc dữ liệu
data = pandas.read_csv('/kaggle/input/nesarc-huit/nesarc.csv', low_memory=False)

In [ ]:
# Chuyển các thuộc tính sang dạng số
data['S2AQ8A'] = pandas.to_numeric(data['S2AQ8A'], errors='coerce')
data['TAB12MDX'] = pandas.to_numeric(data['TAB12MDX'], errors='coerce')

In [ ]:
# Tinh chỉnh dữ liệu
sub1=data[(data['AGE']>=18) & (data['AGE']<=25) & (data['CHECK321']==1)]
sub2 = sub1.copy()

In [ ]:
# Bỏ các dữ liệu thiếu
sub2['S3AQ3B1']=sub2['S3AQ3B1'].replace(9, numpy.nan)
sub2['S3AQ3C1']=sub2['S3AQ3C1'].replace(99, numpy.nan)

In [ ]:
# Số ngày hút thuốc trong tháng
recode2 = {1: 30, 2: 22, 3: 14, 4: 5, 5: 2.5, 6: 1}
sub2['USFREQMO']= sub2['S3AQ3B1'].map(recode2)

# Số điếu thuốc hút trong 1 tháng
sub2['NUMCIGMO_EST']=sub2['USFREQMO'] * sub2['S3AQ3C1']

In [ ]:
######################################################################

In [ ]:
# Đồ thị thể hiện sự phụ thuộc Nicotine trong 12 tháng
sub2["TAB12MDX"] = sub2["TAB12MDX"].astype('category')
seaborn.countplot(x="TAB12MDX", data=sub2)
plt.xlabel('Sự nghiện nicotine trong 12 tháng')
plt.title('Đồ thị thể hiện sự nghiện Nicotine ....')
plt.show()

In [ ]:
# Đồ thị ước lượng số điếu thuốc hút mỗi tháng
seaborn.histplot(data=sub2, x="NUMCIGMO_EST", kde=False)
plt.xlabel('Số điếu thuốc hút mỗi tháng')
plt.ylabel('Số lượng')
plt.title('Ước lượng Số điếu thuốc hút mỗi tháng ......')
plt.show()

In [ ]:
# Thống kê
desc1 = sub2['NUMCIGMO_EST'].describe()
print(desc1)

<h2>Đồ thị với 2 loại</h2>

In [ ]:
sub2['PACKSPERMONTH']=sub2['NUMCIGMO_EST'] / 20
c2= sub2.groupby('PACKSPERMONTH').size()
print(c2)

In [ ]:
sub2['PACKCATEGORY'] = pandas.cut(sub2.PACKSPERMONTH, [0, 5, 10, 20, 30, 147])

In [ ]:
sub2['PACKCATEGORY'] = sub2['PACKCATEGORY'].astype('category')
desc3 = sub2['PACKCATEGORY'].describe()
desc3

In [ ]:
c7 = sub2['PACKCATEGORY'].value_counts(dropna=True).sort_index()
print(c7)

In [ ]:
sub2['TAB12MDX'] = pandas.to_numeric(sub2['TAB12MDX'], errors='coerce')
seaborn.catplot(x="PACKCATEGORY", y="TAB12MDX", data=sub2, kind="bar", errorbar=None)
plt.xlabel('Số gói mỗi tháng')
plt.ylabel('Tỷ lệ lệ thuộc Nicotine')
plt.show()

In [ ]:
# Tạo biến nhóm hút thuốc với 3 loại
def SMOKEGRP(row):
    if row['TAB12MDX'] == 1 :
      return 1
    elif row['USFREQMO'] == 30 :
      return 2
    else :
      return 3
  
sub2['SMOKEGRP'] = sub2.apply(lambda row: SMOKEGRP(row), axis=1)

In [ ]:
c3= sub2['SMOKEGRP'].value_counts(normalize=True).sort_index()
print(c3)

In [ ]:
# Tạo biến hút thuốc hàng ngày
def DAILY (row):
    if row['USFREQMO'] == 30 :
      return 1
    elif row['USFREQMO'] != 30 :
      return 0
      
sub2['DAILY'] = sub2.apply (lambda row: DAILY (row),axis=1)

In [ ]:
sub2['ETHRACE2A'] = sub2['ETHRACE2A'].astype('category')
sub2['ETHRACE2A'] = sub2['ETHRACE2A'].cat.rename_categories(["White", "Black", "NatAm", "Asian", "Hispanic"])

In [ ]:
seaborn.catplot(x='ETHRACE2A', y='DAILY', data=sub2, kind="bar", errorbar=None)
plt.xlabel('Nhóm dân tộc')
plt.ylabel('Tỷ lệ người hút thuốc hàng ngày')
plt.show()